# CrewAI Crash Course

A hands-on introduction to building multi-agent AI systems with CrewAI.

---

## Table of Contents

1. What is CrewAI?
2. Installation and Setup
3. Core Concepts
4. Your First Agent
5. Creating Tasks
6. Building a Crew
7. Tools
8. Real-World Example: Research and Writing Crew
9. Advanced: Custom Tools and Memory
10. Best Practices and Tips

---
## 1. What is CrewAI?

CrewAI is a Python framework for orchestrating role-playing autonomous AI agents that collaborate to accomplish complex tasks.

Think of it like assembling a team of specialists. A Researcher gathers information, a Writer crafts content, and an Editor reviews and polishes the final output. Each agent has a defined role, a goal, and a backstory, and they work together through a Crew.

### Key Features

- Role-based agent design with natural language configuration
- Agents can autonomously delegate tasks to each other
- Built-in support for tools (web search, file I/O, APIs, code execution, etc.)
- Compatible with OpenAI, Anthropic, Google, and local LLMs via Ollama
- Sequential and hierarchical process execution modes
- Memory and context persistence across tasks

---
## 2. Installation and Setup

In [ ]:
# Install CrewAI with the tools extras package
!pip install crewai crewai-tools

In [ ]:
import os

# Set your LLM provider API key
# In production, load these from a .env file or secret manager
os.environ["OPENAI_API_KEY"] = "your-openai-api-key-here"

# To use Anthropic Claude instead:
# os.environ["ANTHROPIC_API_KEY"] = "your-anthropic-api-key-here"

print("Environment configured.")

---
## 3. Core Concepts

CrewAI revolves around four building blocks:

| Concept | Description |
|---------|-------------|
| **Agent** | An autonomous entity with a role, goal, backstory, and optional tools |
| **Task** | A specific piece of work assigned to an agent, with a clear description and expected output |
| **Tool** | A capability an agent can use, such as web search, file reading, or a custom API call |
| **Crew** | The orchestrator that manages agents and tasks, running them in a defined process |

### Process Types

- **Sequential**: Tasks execute one after another. Each task receives the output of the previous task as context.
- **Hierarchical**: A manager agent delegates tasks to worker agents and synthesizes results. Requires a manager LLM.

---
## 4. Your First Agent

In [ ]:
from crewai import Agent

# Define a simple researcher agent
researcher = Agent(
    role="Senior Research Analyst",
    goal="Uncover accurate, well-sourced information on any given topic",
    backstory=(
        "You are a meticulous research analyst with a decade of experience "
        "synthesizing complex topics into clear, evidence-based summaries. "
        "You are thorough, skeptical of unverified claims, and always cite sources."
    ),
    verbose=True,            # Print the agent's reasoning steps
    allow_delegation=False,  # This agent cannot hand off tasks
)

print(researcher)

### Key Agent Parameters

| Parameter | Type | Description |
|-----------|------|-------------|
| `role` | str | The agent's job title, which shapes how it responds |
| `goal` | str | What the agent is ultimately trying to achieve |
| `backstory` | str | Context and personality. The more specific, the better |
| `tools` | list | List of Tool objects the agent can use |
| `llm` | LLM | Override the default LLM for this agent |
| `verbose` | bool | Whether to stream the agent's thought process |
| `allow_delegation` | bool | Whether the agent can delegate subtasks to others |
| `max_iter` | int | Maximum reasoning iterations before stopping (default: 25) |
| `memory` | bool | Whether the agent retains memory across tasks |

---
## 5. Creating Tasks

In [ ]:
from crewai import Task

research_task = Task(
    description=(
        "Research the current state of large language model (LLM) benchmarking. "
        "Cover what major benchmarks exist, their limitations, and recent developments "
        "in evaluation methodology as of 2024."
    ),
    expected_output=(
        "A structured summary of at least 500 words covering the main LLM benchmarks "
        "(e.g., MMLU, HumanEval, HELM), their strengths and weaknesses, and "
        "emerging trends in evaluation."
    ),
    agent=researcher,
)

print(research_task)

### Key Task Parameters

| Parameter | Type | Description |
|-----------|------|-------------|
| `description` | str | What needs to be done. Be specific |
| `expected_output` | str | What a good result looks like |
| `agent` | Agent | Which agent is responsible for this task |
| `tools` | list | Override agent tools for this specific task |
| `context` | list | Other tasks whose outputs feed into this task |
| `output_file` | str | Save the output to a file path |
| `output_pydantic` | BaseModel | Enforce structured JSON output via a Pydantic model |
| `human_input` | bool | Pause and ask for human feedback before finishing |

---
## 6. Building a Crew

In [ ]:
from crewai import Agent, Task, Crew, Process

# Agents
researcher = Agent(
    role="Senior Research Analyst",
    goal="Find accurate, thorough information on any given topic",
    backstory="A meticulous analyst with expertise in synthesizing complex subjects.",
    verbose=True,
    allow_delegation=False,
)

writer = Agent(
    role="Content Writer",
    goal="Transform research findings into clear, engaging prose",
    backstory="An experienced writer who turns dense research into readable articles.",
    verbose=True,
    allow_delegation=False,
)

# Tasks
research_task = Task(
    description="Research the history and impact of the transformer architecture in AI.",
    expected_output="A detailed factual summary covering origins, key papers, and real-world impact.",
    agent=researcher,
)

writing_task = Task(
    description=(
        "Using the research provided, write a 400-word blog post about the transformer "
        "architecture. Target a technically literate but non-specialist audience."
    ),
    expected_output="A polished, publication-ready blog post in markdown format.",
    agent=writer,
    context=[research_task],  # The writer receives the researcher's output
)

# Crew
crew = Crew(
    agents=[researcher, writer],
    tasks=[research_task, writing_task],
    process=Process.sequential,
    verbose=True,
)

result = crew.kickoff()
print(result)

---
## 7. Tools

Tools give agents the ability to interact with the outside world. CrewAI ships with a set of built-in tools via the `crewai-tools` package, and you can also define your own.

In [ ]:
from crewai_tools import (
    SerperDevTool,       # Google search via Serper API
    WebsiteSearchTool,   # RAG-based website content search
    FileReadTool,        # Read local files
    DirectoryReadTool,   # List and read directory contents
    CodeInterpreterTool, # Execute Python code
)

# Initialize a web search tool (requires SERPER_API_KEY env var)
search_tool = SerperDevTool()

# Attach tools to an agent
researcher_with_tools = Agent(
    role="Senior Research Analyst",
    goal="Find accurate information using web search",
    backstory="A thorough analyst who always verifies facts online.",
    tools=[search_tool],
    verbose=True,
)

In [ ]:
from crewai.tools import BaseTool
from pydantic import BaseModel, Field
from typing import Type

# Define the input schema for the custom tool
class WordCountInput(BaseModel):
    text: str = Field(..., description="The text to count words in")

# Build the custom tool by extending BaseTool
class WordCountTool(BaseTool):
    name: str = "Word Counter"
    description: str = "Counts the number of words in a given piece of text."
    args_schema: Type[BaseModel] = WordCountInput

    def _run(self, text: str) -> str:
        count = len(text.split())
        return f"The text contains {count} words."

# Instantiate and test
word_counter = WordCountTool()
print(word_counter.run({"text": "CrewAI makes building multi-agent systems straightforward."}))

---
## 8. Real-World Example: Research and Writing Crew

This example builds a three-agent crew that researches a topic, writes an article, and then edits it. Web search is enabled for the researcher.

In [ ]:
from crewai import Agent, Task, Crew, Process
from crewai_tools import SerperDevTool
import os

# os.environ["OPENAI_API_KEY"] = "..."
# os.environ["SERPER_API_KEY"] = "..."  # Free key at serper.dev

search_tool = SerperDevTool()

# Agents
researcher = Agent(
    role="Research Specialist",
    goal="Gather comprehensive, factual information on the assigned topic",
    backstory=(
        "You are a rigorous research specialist who has spent years fact-checking "
        "complex topics. You prioritize primary sources and always look for multiple "
        "perspectives before drawing conclusions."
    ),
    tools=[search_tool],
    verbose=True,
    allow_delegation=False,
)

writer = Agent(
    role="Technical Writer",
    goal="Write clear, accurate, and engaging articles based on research",
    backstory=(
        "You are a technical writer who specializes in making complex subjects "
        "accessible to a general audience without sacrificing accuracy."
    ),
    verbose=True,
    allow_delegation=False,
)

editor = Agent(
    role="Senior Editor",
    goal="Polish articles for clarity, flow, and factual consistency",
    backstory=(
        "You are a veteran editor with a sharp eye for logical inconsistencies, "
        "unclear phrasing, and structural issues. Your edits improve readability "
        "without changing the core content."
    ),
    verbose=True,
    allow_delegation=False,
)

# Tasks
TOPIC = "How quantum computing threatens current encryption standards"

research_task = Task(
    description=f"Research the following topic thoroughly: {TOPIC}",
    expected_output=(
        "A structured research brief with key facts, relevant data, expert opinions, "
        "and important context. Include any limitations or open questions."
    ),
    agent=researcher,
)

writing_task = Task(
    description=(
        f"Write a 600-word article about: {TOPIC}. "
        "Target audience: educated non-specialists. "
        "Format: markdown with a title, introduction, 3-4 sections, and conclusion."
    ),
    expected_output="A complete markdown article, ready for editorial review.",
    agent=writer,
    context=[research_task],
)

editing_task = Task(
    description=(
        "Review the drafted article for clarity, accuracy, flow, and tone. "
        "Fix any awkward sentences, ensure consistent terminology, and verify "
        "that technical claims are explained clearly."
    ),
    expected_output="The final, polished version of the article in markdown format.",
    agent=editor,
    context=[writing_task],
    output_file="final_article.md",
)

# Crew
crew = Crew(
    agents=[researcher, writer, editor],
    tasks=[research_task, writing_task, editing_task],
    process=Process.sequential,
    verbose=True,
)

result = crew.kickoff()
print("\n=== FINAL OUTPUT ===")
print(result)

---
## 9. Advanced: Custom Tools and Memory

### 9.1 Using a Different LLM

You can configure any agent to use a specific model, including Anthropic Claude or a locally hosted model via Ollama.

In [ ]:
from crewai import Agent, LLM
import os

# Use Claude via Anthropic
claude_llm = LLM(
    model="anthropic/claude-3-5-sonnet-20241022",
    api_key=os.environ.get("ANTHROPIC_API_KEY"),
    temperature=0.3,
)

# Use a local Ollama model
ollama_llm = LLM(
    model="ollama/llama3.1",
    base_url="http://localhost:11434",
)

agent_with_claude = Agent(
    role="Analyst",
    goal="Provide nuanced analysis",
    backstory="A careful and thoughtful analyst.",
    llm=claude_llm,
)

### 9.2 Enabling Memory

Memory allows agents to remember facts across task executions within the same crew run. CrewAI supports short-term (in-context), long-term (vector store), and entity memory.

In [ ]:
from crewai import Crew, Process

crew_with_memory = Crew(
    agents=[researcher, writer],
    tasks=[research_task, writing_task],
    process=Process.sequential,
    memory=True,  # Enable all memory types
    embedder={
        "provider": "openai",
        "config": {"model": "text-embedding-3-small"}
    },
    verbose=True,
)

### 9.3 Structured Output with Pydantic

Enforce a specific output schema by passing a Pydantic model to a task. This is useful when downstream code needs to parse the agent's output programmatically.

In [ ]:
from pydantic import BaseModel
from typing import List
from crewai import Task

class ResearchBrief(BaseModel):
    title: str
    summary: str
    key_points: List[str]
    sources: List[str]
    confidence_level: str  # "high", "medium", or "low"

structured_task = Task(
    description="Research the latest developments in solid-state batteries.",
    expected_output="A structured research brief.",
    agent=researcher,
    output_pydantic=ResearchBrief,
)

# After kickoff, access fields directly:
# result = crew.kickoff()
# print(result.pydantic.key_points)

### 9.4 Hierarchical Process

In hierarchical mode, CrewAI automatically creates a manager agent that plans and delegates tasks to the worker agents. You supply the manager LLM.

In [ ]:
from crewai import Crew, Process, LLM

manager_llm = LLM(model="gpt-4o", temperature=0)

hierarchical_crew = Crew(
    agents=[researcher, writer, editor],
    tasks=[research_task, writing_task, editing_task],
    process=Process.hierarchical,
    manager_llm=manager_llm,
    verbose=True,
)

---
## 10. Best Practices and Tips

### Agent Design

Keep roles focused and specific. An agent that does one thing well outperforms a generalist. Write backstories as if onboarding a new employee — the more context you provide, the more consistent the behavior. Set `allow_delegation=True` only for agents that genuinely need to hand off work.

### Task Design

The `description` tells the agent what to do; the `expected_output` tells it what success looks like. Both matter equally. Use `context` to wire task outputs together rather than repeating information in descriptions. Use `output_file` to persist long outputs, especially when they feed into downstream tasks.

### Crew Design

Start with `Process.sequential` — it is easier to debug and reason about. Move to `Process.hierarchical` when task routing needs to be dynamic or when the number of tasks is large. Use `verbose=True` during development and disable it in production.

### Cost and Performance

Each agent call consumes tokens. Use cheaper models (GPT-4o-mini, Claude Haiku) for simpler agents and reserve expensive models for complex reasoning tasks. Set `max_iter` to prevent agents from looping indefinitely. Cache tool calls where possible to reduce repeated API requests.

### Debugging

Run with `verbose=True` and read the chain-of-thought output carefully. If an agent fails repeatedly, tighten the task description and expected output. Use `human_input=True` on a task to pause execution and inspect intermediate results interactively.

In [ ]:
cheatsheet = """
CrewAI Quick Reference
======================

from crewai import Agent, Task, Crew, Process, LLM
from crewai_tools import SerperDevTool

# Agent
agent = Agent(
    role="...",
    goal="...",
    backstory="...",
    tools=[SerperDevTool()],
    llm=LLM(model="gpt-4o"),
    verbose=True,
    allow_delegation=True,
    memory=True,
    max_iter=15,
)

# Task
task = Task(
    description="...",
    expected_output="...",
    agent=agent,
    context=[other_task],
    output_file="result.md",
    output_pydantic=MyModel,
    human_input=False,
)

# Crew
crew = Crew(
    agents=[agent],
    tasks=[task],
    process=Process.sequential,   # or Process.hierarchical
    manager_llm=LLM(model="gpt-4o"),  # required for hierarchical
    memory=True,
    verbose=True,
)

# Run
result = crew.kickoff(inputs={"topic": "AI safety"})
print(result.raw)          # Raw string output
print(result.pydantic)     # Pydantic model (if output_pydantic was set)
print(result.token_usage)  # Token consumption stats
"""

print(cheatsheet)